In [1]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

## Image segmentation with SAM 3

This notebook demonstrates how to use SAM 3 for image segmentation with text or visual prompts. It covers the following capabilities:

- **Text prompts**: Using natural language descriptions to segment objects (e.g., "person", "face")
- **Box prompts**: Using bounding boxes as exemplar visual prompts

# <a target="_blank" href="https://colab.research.google.com/github/facebookresearch/sam3/blob/main/notebooks/sam3_image_predictor_example.ipynb">
#   <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
# </a>

In [6]:
using_colab = True

### 修正 SAM3 安裝問題
由於 Python 3.13 環境中編譯舊版 `numpy` 可能會遭遇相容性問題，我們透過指定不安裝特定舊版 numpy，或是利用 `--no-build-isolation` 強制使用系統現有環境來順利安裝 `sam3`。

In [ ]:
import sys

# 1. 升級基礎建置工具
!{sys.executable} -m pip install --upgrade pip setuptools wheel

# 2. 先安裝 numpy 編譯所需的 meson-python 與 ninja 建置依賴
!{sys.executable} -m pip install meson-python ninja

# 3. 再進行 sam3 的安裝
!{sys.executable} -m pip install 'git+https://github.com/facebookresearch/sam3.git'

  Using cached ninja-1.13.2-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.4 kB)
  Using cached meson-1.12.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached pyproject_metadata-0.12.1-py3-none-any.whl.metadata (6.5 kB)
Using cached ninja-1.13.2-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (183 kB)
Using cached meson-1.12.1-py3-none-any.whl (1.1 MB)
Using cached pyproject_metadata-0.12.1-py3-none-any.whl (23 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [meson-python]
  Cloning https://github.com/facebookresearch/sam3.git to /tmp/pip-req-build-qpq2zrc0
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/sam3.git /tmp/pip-req-build-qpq2zrc0
  Resolved https://github.com/facebookresearch/sam3.git to commit 0570b3a5be9c4e694f23d85232fb55f4a6f1f7fc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-1.26.4.tar.g

In [7]:
if using_colab:
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    import sys
    !{sys.executable} -m pip install opencv-python matplotlib scikit-learn
    !{sys.executable} -m pip install 'git+https://github.com/facebookresearch/sam3.git'

PyTorch version: 2.11.0+cu130
Torchvision version: 0.26.0+cu130
CUDA is available: True
  Cloning https://github.com/facebookresearch/sam3.git to /tmp/pip-req-build-fw5qznt0
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/sam3.git /tmp/pip-req-build-fw5qznt0
  Resolved https://github.com/facebookresearch/sam3.git to commit 0570b3a5be9c4e694f23d85232fb55f4a6f1f7fc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... canceledERROR: Operation cancelled by user


In [8]:
import os
import sys

# 自動偵測並安裝 sam3 套件以避免 ModuleNotFoundError
try:
    import sam3
except ModuleNotFoundError:
    print("偵測到未安裝 sam3，正在進行安裝...")
    !{sys.executable} -m pip install opencv-python matplotlib scikit-learn
    !{sys.executable} -m pip install 'git+https://github.com/facebookresearch/sam3.git'
    import sam3

import matplotlib.pyplot as plt
import numpy as np

from PIL import Image
from sam3 import build_sam3_image_model
from sam3.model.box_ops import box_xywh_to_cxcywh
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.visualization_utils import draw_box_on_image, normalize_bbox, plot_results

sam3_root = os.path.join(os.path.dirname(sam3.__file__), "..")

偵測到未安裝 sam3，正在進行安裝...
  Cloning https://github.com/facebookresearch/sam3.git to /tmp/pip-req-build-racbpc0i
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/sam3.git /tmp/pip-req-build-racbpc0i
  Resolved https://github.com/facebookresearch/sam3.git to commit 0570b3a5be9c4e694f23d85232fb55f4a6f1f7fc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... canceled
ERROR: Operation cancelled by user


ModuleNotFoundError: No module named 'sam3'

In [ ]:
import torch

# turn on tfloat32 for Ampere GPUs
# https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# use bfloat16 for the entire notebook
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

# Build Model

In [ ]:
bpe_path = f"{sam3_root}/assets/bpe_simple_vocab_16e6.txt.gz"
model = build_sam3_image_model(bpe_path=bpe_path)

In [ ]:
image_path = f"{sam3_root}/assets/images/test_image.jpg"
image = Image.open(image_path)
width, height = image.size
processor = Sam3Processor(model, confidence_threshold=0.5)
inference_state = processor.set_image(image)

### 載入房屋影像測試
我們從網路下載一張常見的房屋外觀影像，並將其載入至 SAM3 處理器中。

In [ ]:
import requests
from io import BytesIO

# 房屋範例影像網址
house_image_url = "https://images.unsplash.com/photo-1570129477492-45c003edd2be?w=800&auto=format&fit=crop"

try:
    response = requests.get(house_image_url)
    house_image = Image.open(BytesIO(response.content)).convert("RGB")

    # 更新影像路徑與處理器狀態
    image = house_image
    width, height = image.size
    inference_state = processor.set_image(image)

    # 顯示載入的影像大小與預覽
    print(f"成功載入房屋影像！解析度為: {width}x{height}")
    display(image.resize((400, 300)))
except Exception as e:
    print("影像下載或載入失敗，請確認網路連線：", e)

載入成功後，您可以直接使用下方的 **Text prompt**（例如將提示詞修改為 `"house"`、`"roof"` 或 `"window"`）或 **Visual prompt** 來進行房屋區域的分割測試！

### 2. 上傳本機房屋影像進行測試
執行下方單元格，並點擊「選擇檔案（Choose Files）」按鈕，即可上傳您自己的房屋照片。

In [ ]:
from google.colab import files
import io

# 啟動 Colab 檔案上傳介面
uploaded = files.upload()

if uploaded:
    # 取得上傳的第一個檔案名稱
    file_name = list(uploaded.keys())[0]
    print(f"成功上傳檔案: {file_name}")

    # 將上傳的檔案轉為 PIL Image 格式
    image = Image.open(io.BytesIO(uploaded[file_name])).convert("RGB")
    width, height = image.size

    # 將新影像設定至 SAM3 處理器中
    inference_state = processor.set_image(image)

    # 顯示上傳影像預覽
    print(f"已將 SAM3 輸入影像更新為本機圖片！解析度: {width}x{height}")
    display(image.resize((400, int(400 * height / width))))
else:
    print("未選取任何檔案，請重新執行單元格。")

# Text prompt

In [ ]:
processor.reset_all_prompts(inference_state)
inference_state = processor.set_text_prompt(state=inference_state, prompt="shoe")

img0 = Image.open(image_path)
plot_results(img0, inference_state)

### Visual prompt: a single bounding box

In [ ]:
# Here the box is in  (x,y,w,h) format, where (x,y) is the top left corner.
box_input_xywh = torch.tensor([480.0, 290.0, 110.0, 360.0]).view(-1, 4)
box_input_cxcywh = box_xywh_to_cxcywh(box_input_xywh)

norm_box_cxcywh = normalize_bbox(box_input_cxcywh, width, height).flatten().tolist()
print("Normalized box input:", norm_box_cxcywh)

processor.reset_all_prompts(inference_state)
inference_state = processor.add_geometric_prompt(
    state=inference_state, box=norm_box_cxcywh, label=True
)

img0 = Image.open(image_path)
image_with_box = draw_box_on_image(img0, box_input_xywh.flatten().tolist())
plt.imshow(image_with_box)
plt.axis("off")  # Hide the axis
plt.show()

In [ ]:
plot_results(img0, inference_state)

### Visual prompt: multi-box prompting (with positive and negative boxes)

In [ ]:
box_input_xywh = [[480.0, 290.0, 110.0, 360.0], [370.0, 280.0, 115.0, 375.0]]
box_input_cxcywh = box_xywh_to_cxcywh(torch.tensor(box_input_xywh).view(-1,4))
norm_boxes_cxcywh = normalize_bbox(box_input_cxcywh, width, height).tolist()

box_labels = [True, False]

processor.reset_all_prompts(inference_state)

for box, label in zip(norm_boxes_cxcywh, box_labels):
    inference_state = processor.add_geometric_prompt(
        state=inference_state, box=box, label=label
    )

img0 = Image.open(image_path)
image_with_box = img0
for i in range(len(box_input_xywh)):
    if box_labels[i] == 1:
        color = (0, 255, 0)
    else:
        color = (255, 0, 0)
    image_with_box = draw_box_on_image(image_with_box, box_input_xywh[i], color)
plt.imshow(image_with_box)
plt.axis("off")  # Hide the axis
plt.show()

In [ ]:
plot_results(img0, inference_state)